# AL Test Generator — Agente RAG híbrido para Business Central

Este notebook recorre paso a paso todo lo que hace el agente: desde el libro de testing en AL hasta la generación de un codeunit de tests completo.

## Arquitectura general

```
libro_al_testing.md          guarantees/src/*.al       guarantees-test/**/*.al
       │                             │                           │
       ▼                             ▼                           ▼
  parser.py                    al_parser.py               al_parser.py
  chunker.py                  (app_code)              (company_example)
       │                             │                           │
       └─────────────────────────────┴───────────────────────────┘
                                     │
                                     ▼
                             embedder.py (BGE-M3)
                                     │
                                     ▼
                             ChromaDB vectorstore
                                     │
                    ┌────────────────┴─────────────────┐
                    ▼                                   ▼
             Python retrieval                    LLM (hasta 2 tool calls extra)
          (RAG + fichero fuente                        │
           + test de referencia)                       │
                    └────────────────┬─────────────────┘
                                     ▼
                              Prompt compacto
                                     │
                                     ▼
                              LLM → codeunit AL
```

## 1. Configuración del entorno

In [ ]:
import os, sys

# Evitar timeouts de HuggingFace (modelo ya cacheado localmente)
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# API keys — define las tuyas aquí o en variables de entorno
# os.environ["OPENAI_API_KEY"]     = "sk-..."
# os.environ["ANTHROPIC_API_KEY"]  = "sk-ant-..."

print("Entorno configurado.")

## 2. Paso 1 — Parseo del libro de testing en AL

Lee el markdown del libro y agrupa el contenido por sección. Cada sección queda clasificada con un `content_type`:

| content_type | Contenido |
|---|---|
| `test_example` | Ejemplos de código AL con tests reales |
| `concept` | Explicaciones teóricas (TDD, frameworks…) |
| `summary` | Resúmenes de capítulo |
| `reference` | Bibliografía / further reading |

In [ ]:
import re
from dataclasses import dataclass, field
from collections import Counter

# ── Modelos de datos ─────────────────────────────────────────────────

@dataclass
class Section:
    chapter_num: int
    chapter_title: str
    section_title: str
    content_type: str          # test_example | concept | summary | reference
    pages: list = field(default_factory=list)
    content: str = ""

    @property
    def chunk_id(self) -> str:
        safe_section = re.sub(r"[^a-z0-9]+", "_", self.section_title.lower()).strip("_")
        return f"ch{self.chapter_num}_{safe_section}"

    @property
    def page_range(self) -> str:
        if not self.pages:
            return "?"
        return f"{min(self.pages)}\u2013{max(self.pages)}"

    def __repr__(self) -> str:
        return (
            f"Section(ch={self.chapter_num}, "
            f"type={self.content_type!r}, "
            f"pages={self.page_range}, "
            f"title={self.section_title!r})"
        )


# ── Helpers de clasificación ─────────────────────────────────────────

_CONTENT_TYPE_RULES = [
    ("test example",    "test_example"),
    ("summary",         "summary"),
    ("further reading", "reference"),
    ("introduction",    "concept"),
    ("framework",       "concept"),
]

def _detect_content_type(section_title: str) -> str:
    lower = section_title.lower()
    for keyword, ctype in _CONTENT_TYPE_RULES:
        if keyword in lower:
            return ctype
    return "concept"


# Patrones de los headers Markdown
_CHAPTER_RE = re.compile(r"^# (\d+)\s+(.+)$")
_SECTION_RE = re.compile(r"^## (.+?) - Page: (\d+)$")


# ── Parser principal ──────────────────────────────────────────────────

def parse_markdown(md_text: str) -> list:
    """
    Recorre el markdown línea a línea y agrupa el contenido por sección.
    Devuelve una lista de Section ordenadas por aparición en el documento.
    """
    sections = []
    current_chapter_num   = 0
    current_chapter_title = ""
    current_section       = None

    for raw_line in md_text.splitlines():
        line = raw_line.strip()

        ch_match = _CHAPTER_RE.match(line)
        if ch_match:
            current_chapter_num   = int(ch_match.group(1))
            current_chapter_title = ch_match.group(2).strip()
            continue

        sec_match = _SECTION_RE.match(line)
        if sec_match:
            sec_title = sec_match.group(1).strip()
            page_num  = int(sec_match.group(2))

            if current_section and current_section.section_title == sec_title:
                current_section.pages.append(page_num)
            else:
                if current_section:
                    sections.append(current_section)
                current_section = Section(
                    chapter_num    = current_chapter_num,
                    chapter_title  = current_chapter_title,
                    section_title  = sec_title,
                    content_type   = _detect_content_type(sec_title),
                    pages          = [page_num],
                )
            continue

        if current_section and line:
            current_section.content += raw_line + "\n"

    if current_section:
        sections.append(current_section)

    return sections


# ── Uso ───────────────────────────────────────────────────────────────

LIBRO_PATH = "../libro_al_testing.md"

with open(LIBRO_PATH, encoding="utf-8") as f:
    md_text = f.read()

sections = parse_markdown(md_text)

type_counts = Counter(s.content_type for s in sections)
print(f"Total secciones: {len(sections)}\n")
print("Por content_type:")
for t, c in sorted(type_counts.items()):
    print(f"  {t:15s}: {c}")

print("\nMuestra (primeras 5):")
for s in sections[:5]:
    words = len(s.content.split())
    print(f"  [{s.content_type:12s}] ch{s.chapter_num} | p.{s.page_range:>8s} | {words:>4} palabras | {s.section_title}")

## 3. Paso 2 — Chunking del libro

Divide cada sección en chunks de ~2400 caracteres respetando los límites naturales de AL (`procedure`, `begin`, `end`). Cada chunk se convierte en un `Document` de LangChain con metadatos enriquecidos.

In [ ]:
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Configuración del splitter ────────────────────────────────────────

# Separadores en orden de preferencia — respetan la sintaxis de AL
AL_SEPARATORS = [
    "\nprocedure ",
    "\nlocal procedure ",
    "\ntrigger ",
    "\nbegin\n",
    "\nend;\n",
    "\nend.\n",
    "\n\n",
    "\n",
    " ",
]

CHUNK_SIZE    = 2_400   # caracteres (~600-700 tokens para BGE-M3)
CHUNK_OVERLAP = 240     # ~10% — suficiente para no perder contexto entre chunks

_splitter = RecursiveCharacterTextSplitter(
    chunk_size      = CHUNK_SIZE,
    chunk_overlap   = CHUNK_OVERLAP,
    separators      = AL_SEPARATORS,
    length_function = len,
    is_separator_regex=False,
)


# ── Conversión Section → Documents ───────────────────────────────────

def section_to_documents(section) -> list:
    """
    Convierte una Section en uno o varios Documents con metadata enriquecida.
    El prefijo jerárquico se antepone a cada chunk para que el LLM siempre
    sepa de qué capítulo/sección proviene el contexto recuperado.
    """
    if not section.content.strip():
        return []

    header = (
        f"# Chapter {section.chapter_num}: {section.chapter_title}\n"
        f"## Section: {section.section_title}\n"
        f"Pages: {section.page_range}\n"
        "---\n"
    )

    full_text  = header + section.content
    raw_chunks = _splitter.split_text(full_text)

    docs = []
    for idx, chunk in enumerate(raw_chunks):
        docs.append(Document(
            page_content=chunk,
            metadata={
                "chunk_id":      f"{section.chunk_id}__{idx}",
                "chunk_index":   idx,
                "chunk_total":   len(raw_chunks),
                "chapter_num":   section.chapter_num,
                "chapter_title": section.chapter_title,
                "section_title": section.section_title,
                "content_type":  section.content_type,
                "page_start":    min(section.pages) if section.pages else 0,
                "page_end":      max(section.pages) if section.pages else 0,
            },
        ))

    return docs


def sections_to_documents(sections: list) -> list:
    """Convierte una lista de Section en todos sus Documents."""
    all_docs = []
    for section in sections:
        all_docs.extend(section_to_documents(section))
    return all_docs


# ── Uso ───────────────────────────────────────────────────────────────

book_docs = sections_to_documents(sections)

word_counts = [len(d.page_content.split()) for d in book_docs]
print(f"Total chunks del libro : {len(book_docs)}")
print(f"Palabras por chunk     : min={min(word_counts)}, media={sum(word_counts)//len(word_counts)}, max={max(word_counts)}")

example = next(d for d in book_docs if d.metadata["content_type"] == "test_example")
print(f"\nEjemplo de chunk (test_example):")
print(f"  chunk_id    : {example.metadata['chunk_id']}")
print(f"  chapter     : {example.metadata['chapter_num']} \u2014 {example.metadata['chapter_title']}")
print(f"  section     : {example.metadata['section_title']}")
print(f"  pages       : {example.metadata['page_start']}\u2013{example.metadata['page_end']}")
print(f"\nPrimeros 300 chars del contenido:")
print(example.page_content[:300])

## 4. Paso 3 — Parseo de los archivos AL del proyecto

Lee los ficheros `.al` de la extensión GarantiasAAPP y genera chunks por procedimiento/trigger. Cada fichero queda clasificado según su carpeta:

| content_type | Carpeta | Contenido |
|---|---|---|
| `app_code` | `guarantees/src/` | Tablas, páginas, codeunits de la app |
| `company_example` | `guarantees-test/` | Tests reales del proyecto |

In [ ]:
from pathlib import Path
from langchain.text_splitter import RecursiveCharacterTextSplitter

# ── Configuración ────────────────────────────────────────────────────

AL_CHUNK_SIZE    = 2_400
AL_CHUNK_OVERLAP = 240

_FALLBACK_SPLITTER = RecursiveCharacterTextSplitter(
    chunk_size         = AL_CHUNK_SIZE,
    chunk_overlap      = AL_CHUNK_OVERLAP,
    separators         = ["\nbegin\n", "\nend;\n", "\n\n", "\n", " "],
    length_function    = len,
    is_separator_regex = False,
)


# ── Patrones de expresiones regulares ────────────────────────────────

_OBJECT_RE = re.compile(
    r'^\s*(?P<obj_type>codeunit|table|page|report|xmlport|query|enum|interface|'
    r'tableextension|pageextension|reportextension|enumextension|'
    r'permissionset|permissionsetextension|profile|controladdin)\s+'
    r'(?P<obj_id>\d+)\s+'
    r'(?:"(?P<name_q>[^"]+)"|(?P<name_u>[A-Za-z_][A-Za-z0-9_ ]*))',
    re.IGNORECASE | re.MULTILINE,
)

_PROC_RE = re.compile(
    r'^[ \t]{0,12}'
    r'(?:local\s+|internal\s+|protected\s+)?'
    r'(?P<kind>procedure|trigger)\s+'
    r'(?:"(?P<name_q>[^"]+)"|(?P<name_u>[A-Za-z_][A-Za-z0-9_]*))',
    re.IGNORECASE | re.MULTILINE,
)

_SUBTYPE_TEST_RE = re.compile(r'Subtype\s*=\s*Test\s*;', re.IGNORECASE)
_TEST_ATTR_RE    = re.compile(r'\[Test\]',                re.IGNORECASE)


# ── Lógica principal ─────────────────────────────────────────────────

def _al_prefix(object_type: str, object_id: int, object_name: str,
               proc_name) -> str:
    """Cabecera contextual prepuesta a cada chunk."""
    line = f"// [{object_type} {object_id}] {object_name}"
    if proc_name:
        line += f" \u2192 {proc_name}"
    return line + "\n"


def parse_al_file(filepath: Path, is_test_dir: bool) -> list:
    """
    Parsea un único fichero .al y devuelve sus Documents.
    Args:
        filepath:    Ruta absoluta al fichero .al.
        is_test_dir: True si la carpeta está clasificada como tests.
    """
    try:
        text = filepath.read_text(encoding="utf-8", errors="replace")
    except OSError:
        return []

    obj_match = _OBJECT_RE.search(text)
    if not obj_match:
        return []

    object_type  = obj_match.group("obj_type").lower()
    object_id    = int(obj_match.group("obj_id"))
    object_name  = (obj_match.group("name_q") or obj_match.group("name_u") or "").strip()

    is_test_object = bool(_SUBTYPE_TEST_RE.search(text))
    content_type   = "company_example" if (is_test_dir or is_test_object) else "app_code"
    source_file    = filepath.name

    proc_matches = list(_PROC_RE.finditer(text))
    sections_al  = []  # (proc_name, is_test_proc, raw_text)

    if not proc_matches:
        sections_al.append((None, False, text))
    else:
        header_raw = text[:proc_matches[0].start()].rstrip()
        if header_raw.strip():
            sections_al.append((None, False, header_raw))

        for idx, m in enumerate(proc_matches):
            end      = proc_matches[idx + 1].start() if idx + 1 < len(proc_matches) else len(text)
            proc_raw = text[m.start():end].rstrip()

            proc_name = (m.group("name_q") or m.group("name_u") or "Unknown").strip()

            before       = text[max(0, m.start() - 200): m.start()]
            is_test_proc = bool(_TEST_ATTR_RE.search(before))

            sections_al.append((proc_name, is_test_proc, proc_raw))

    docs = []
    for proc_name, is_test_proc, raw in sections_al:
        full_text = _al_prefix(object_type, object_id, object_name, proc_name) + raw

        if len(full_text) > AL_CHUNK_SIZE + 200:
            sub_chunks = _FALLBACK_SPLITTER.split_text(full_text)
        else:
            sub_chunks = [full_text]

        for sub_idx, chunk_text in enumerate(sub_chunks):
            safe_obj  = re.sub(r"[^a-z0-9]+", "_", object_name.lower()).strip("_")
            safe_proc = re.sub(r"[^a-z0-9]+", "_", (proc_name or "header").lower()).strip("_")
            chunk_id  = f"{object_type}_{safe_obj}__{safe_proc}__{sub_idx}"

            docs.append(Document(
                page_content=chunk_text,
                metadata={
                    "chunk_id":          chunk_id,
                    "chunk_index":       sub_idx,
                    "chunk_total":       len(sub_chunks),
                    "content_type":      content_type,
                    "object_type":       object_type,
                    "object_id":         object_id,
                    "object_name":       object_name,
                    "procedure_name":    proc_name or "",
                    "is_test_object":    is_test_object,
                    "is_test_procedure": is_test_proc,
                    "source_file":       source_file,
                },
            ))

    return docs


def parse_al_directory(src_dirs: list, test_dirs: list) -> list:
    """
    Escanea directorios de fuentes y de tests y parsea todos los .al.
    Args:
        src_dirs:  Directorios con código de la app (content_type = 'app_code').
        test_dirs: Directorios con tests (content_type = 'company_example').
    """
    all_docs = []

    for directory in src_dirs:
        for filepath in sorted(directory.rglob("*.al")):
            all_docs.extend(parse_al_file(filepath, is_test_dir=False))

    for directory in test_dirs:
        for filepath in sorted(directory.rglob("*.al")):
            all_docs.extend(parse_al_file(filepath, is_test_dir=True))

    return all_docs


# ── Uso ───────────────────────────────────────────────────────────────

PROJECT_ROOT = Path(r"C:\Users\mscarafu\Downloads\GarantiasAAPP")

src_dirs  = [PROJECT_ROOT / "guarantees" / "src"]
test_dirs = [
    PROJECT_ROOT / "guarantees-test" / "src" / "codeunit",
    PROJECT_ROOT / "guarantees-test" / "test",
]

al_docs = parse_al_directory(src_dirs=src_dirs, test_dirs=test_dirs)

type_counts_al = Counter(d.metadata["content_type"] for d in al_docs)
obj_counts     = Counter(d.metadata["object_type"]  for d in al_docs)

print(f"Total chunks AL: {len(al_docs)}\n")
print("Por content_type:")
for t, c in sorted(type_counts_al.items()):
    print(f"  {t:20s}: {c}")
print("\nPor object_type:")
for t, c in sorted(obj_counts.items()):
    print(f"  {t:20s}: {c}")

example_al = next(d for d in al_docs if d.metadata["content_type"] == "company_example"
                  and d.metadata["procedure_name"])
print(f"\nEjemplo chunk company_example:")
print(f"  object : {example_al.metadata['object_type']} \u2014 {example_al.metadata['object_name']}")
print(f"  proc   : {example_al.metadata['procedure_name']}")
print(f"  file   : {example_al.metadata['source_file']}")
print(f"\nPrimeros 300 chars:")
print(example_al.page_content[:300])

## 5. Paso 4 — Embeddings y vectorstore (ChromaDB)

Genera embeddings e ingesta todos los chunks en ChromaDB. En este notebook usamos **BAAI/bge-small-en-v1.5** (~130 MB) para mayor velocidad. El vectorstore de producción usa **BAAI/bge-m3** (~570 MB, más preciso).

> **Nota:** si el vectorstore de producción ya está construido en `./chroma_al_testing` con bge-m3, no ejecutes la celda de construcción — carga directamente el existente con el modelo correcto.

In [ ]:
import shutil
from pathlib import Path
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from tqdm import tqdm

# ── Configuración ────────────────────────────────────────────────────

EMBEDDING_MODEL  = "BAAI/bge-m3"             # producción
NOTEBOOK_MODEL   = "BAAI/bge-small-en-v1.5"  # demo (más rápido, ~130 MB)
COLLECTION_NAME  = "al_testing_book"
DEFAULT_PERSIST  = "./chroma_al_testing"
INGEST_BATCH_SIZE = 64


# ── Funciones de embedding ────────────────────────────────────────────

def load_embeddings(device: str = "cpu", model_name: str = EMBEDDING_MODEL) -> HuggingFaceEmbeddings:
    """
    Carga el modelo de embedding desde HuggingFace (se cachea localmente).
    Args:
        device:     'cpu' o 'cuda'.
        model_name: Nombre del modelo HuggingFace.
    """
    print(f"[embedder] Cargando modelo {model_name} en {device}...")
    return HuggingFaceEmbeddings(
        model_name   = model_name,
        model_kwargs = {"device": device},
        encode_kwargs= {
            "normalize_embeddings": True,  # obligatorio para BGE
            "batch_size": 32,
        },
    )


def ingest_documents(docs: list, embeddings, persist_dir: str = DEFAULT_PERSIST,
                     overwrite: bool = False) -> Chroma:
    """
    Ingesta Documents en ChromaDB.
    Si persist_dir ya existe y overwrite=False, lanza un error para evitar duplicar vectores.
    """
    persist_path = Path(persist_dir)

    if persist_path.exists() and any(persist_path.iterdir()):
        if not overwrite:
            raise FileExistsError(
                f"'{persist_dir}' ya contiene datos. "
                "Usa overwrite=True para sobreescribir o load_vectorstore() para reutilizarlo."
            )
        print(f"[embedder] overwrite=True — borrando colección existente en '{persist_dir}'...")
        shutil.rmtree(persist_dir)

    print(f"[embedder] Ingestando {len(docs)} chunks en '{persist_dir}'...")

    vectorstore = None
    batches = [docs[i:i + INGEST_BATCH_SIZE] for i in range(0, len(docs), INGEST_BATCH_SIZE)]

    for batch in tqdm(batches, desc="Embedding batches"):
        if vectorstore is None:
            vectorstore = Chroma.from_documents(
                documents         = batch,
                embedding         = embeddings,
                persist_directory = persist_dir,
                collection_name   = COLLECTION_NAME,
            )
        else:
            vectorstore.add_documents(batch)

    print(f"[embedder] ✓ Ingesta completada. {len(docs)} chunks almacenados.")
    return vectorstore


def load_vectorstore(embeddings, persist_dir: str = DEFAULT_PERSIST) -> Chroma:
    """Carga un vectorstore ya persistido en disco sin re-embeber nada."""
    if not Path(persist_dir).exists():
        raise FileNotFoundError(
            f"No se encontró vectorstore en '{persist_dir}'. "
            "Ejecuta primero ingest_documents()."
        )
    print(f"[embedder] Cargando vectorstore desde '{persist_dir}'...")
    return Chroma(
        persist_directory  = persist_dir,
        embedding_function = embeddings,
        collection_name    = COLLECTION_NAME,
    )


def append_documents(docs: list, embeddings, persist_dir: str = DEFAULT_PERSIST) -> Chroma:
    """
    Añade Documents a un vectorstore ya existente sin borrar el contenido previo.
    Si el vectorstore no existe, lo crea desde cero.
    """
    persist_path = Path(persist_dir)

    if persist_path.exists() and any(persist_path.iterdir()):
        print(f"[embedder] Añadiendo {len(docs)} chunks al vectorstore existente en '{persist_dir}'...")
        vectorstore = Chroma(
            persist_directory  = persist_dir,
            embedding_function = embeddings,
            collection_name    = COLLECTION_NAME,
        )
    else:
        print(f"[embedder] Vectorstore no encontrado — creando uno nuevo en '{persist_dir}'...")
        vectorstore = None

    batches = [docs[i:i + INGEST_BATCH_SIZE] for i in range(0, len(docs), INGEST_BATCH_SIZE)]
    for batch in tqdm(batches, desc="Embedding batches"):
        if vectorstore is None:
            vectorstore = Chroma.from_documents(
                documents         = batch,
                embedding         = embeddings,
                persist_directory = persist_dir,
                collection_name   = COLLECTION_NAME,
            )
        else:
            vectorstore.add_documents(batch)

    print(f"[embedder] ✓ {len(docs)} chunks añadidos.")
    return vectorstore


# ── Demo: vectorstore notebook con bge-small ─────────────────────────

PERSIST_DIR_NOTEBOOK = "./chroma_notebook_demo"

print(f"Cargando modelo demo: {NOTEBOOK_MODEL}")
embeddings_small = load_embeddings(device="cpu", model_name=NOTEBOOK_MODEL)
print("Modelo cargado.")

# Total chunks disponibles
all_chunks = book_docs + al_docs
print(f"\nTotal chunks a ingestar: {len(all_chunks)}")
print(f"  Libro : {len(book_docs)}")
print(f"  AL    : {len(al_docs)}")

# Construir el vectorstore demo (solo ejecutar una vez)
# vectorstore_demo = ingest_documents(
#     docs        = all_chunks,
#     embeddings  = embeddings_small,
#     persist_dir = PERSIST_DIR_NOTEBOOK,
#     overwrite   = True,
# )
# print(f"\nVectorstore demo construido en '{PERSIST_DIR_NOTEBOOK}'")

## 6. Paso 5 — Retriever híbrido (vectorial + BM25)

Construye un `EnsembleRetriever` que combina dos estrategias complementarias:

**¿Por qué híbrido?**

- La **búsqueda vectorial sola** es buena para similitud semántica ("testear una contabilización") pero falla con términos técnicos exactos: si busco `DTNGARManagement` puede devolver chunks de otros codeunits que semánticamente se parecen pero no son el correcto.
- El **BM25 solo** encuentra keywords exactas (`LibraryAssert`, `BindSubscription`, nombres de procedimientos AL) pero no entiende el contexto semántico: no sabe que "posting a guarantee" y "contabilizar una garantía" son lo mismo.
- **Combinados** (60% vectorial / 40% BM25): el retriever encuentra a la vez fragmentos conceptualmente relevantes y el código exacto referenciado en la query.

In [ ]:
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

# ── Retriever híbrido ─────────────────────────────────────────────────

def build_hybrid_retriever(vectorstore, all_docs: list, k: int = 6,
                           vector_weight: float = 0.6,
                           content_type_filter = None) -> EnsembleRetriever:
    """
    Combina búsqueda vectorial MMR con BM25 para mayor robustez.

    La búsqueda vectorial captura similitud semántica.
    BM25 captura keywords exactas (nombres de funciones AL, LibraryAssert, etc.).

    Args:
        vectorstore:         Chroma ya cargado.
        all_docs:            Todos los Documents (para BM25).
        k:                   Número de resultados a devolver.
        vector_weight:       Peso del retriever vectorial (BM25 = 1 - vector_weight).
        content_type_filter: Si se especifica, filtra por tipo antes de buscar.
    """
    search_kwargs = {
        "k":           k,
        "fetch_k":     k * 4,   # MMR explora más candidatos antes de diversificar
        "lambda_mult": 0.7,     # 0 = máxima diversidad, 1 = máxima relevancia
    }
    if content_type_filter:
        search_kwargs["filter"] = {"content_type": content_type_filter}

    vector_retriever = vectorstore.as_retriever(
        search_type   = "mmr",
        search_kwargs = search_kwargs,
    )

    bm25_docs = (
        [d for d in all_docs if d.metadata.get("content_type") == content_type_filter]
        if content_type_filter
        else all_docs
    )
    bm25_retriever   = BM25Retriever.from_documents(bm25_docs)
    bm25_retriever.k = k

    return EnsembleRetriever(
        retrievers = [vector_retriever, bm25_retriever],
        weights    = [vector_weight, 1 - vector_weight],
    )


def get_retriever_for_query(vectorstore, all_docs: list, k: int = 6) -> EnsembleRetriever:
    """
    Devuelve un retriever híbrido sin filtro de tipo, para que las queries
    mezclen ejemplos del libro (test_example), tests reales de la empresa
    (company_example) y código de la app (app_code).
    """
    return build_hybrid_retriever(
        vectorstore         = vectorstore,
        all_docs            = all_docs,
        k                   = k,
        content_type_filter = None,
    )


# ── Cargar vectorstore de producción (bge-m3) ─────────────────────────

embeddings_prod = load_embeddings(device="cpu", model_name=EMBEDDING_MODEL)
vectorstore     = load_vectorstore(embeddings_prod, persist_dir=DEFAULT_PERSIST)

raw      = vectorstore.get(include=["documents", "metadatas"])
all_docs = [
    Document(page_content=t, metadata=m)
    for t, m in zip(raw["documents"], raw["metadatas"])
]

print(f"Chunks en el vectorstore de producción: {len(all_docs)}")

type_counts_vs = Counter(d.metadata.get("content_type", "?") for d in all_docs)
print("\nDistribución por content_type:")
for t, c in sorted(type_counts_vs.items()):
    print(f"  {t:20s}: {c}")

In [ ]:
# ── Demostración del retriever híbrido ────────────────────────────────

QUERY = "How to test posting a bank guarantee in AL?"

retriever = get_retriever_for_query(vectorstore, all_docs, k=6)
results   = retriever.invoke(QUERY)

print(f"Query: {QUERY!r}\n")
print(f"{'─'*70}")
for i, doc in enumerate(results, 1):
    meta = doc.metadata
    if "chapter_num" in meta:
        origen = f"[libro] ch{meta['chapter_num']} \u2014 {meta['section_title'][:50]}"
    else:
        proc   = meta.get("procedure_name") or "header"
        origen = f"[{meta.get('content_type')}] {meta.get('object_name')} \u2192 {proc}"
    print(f"\n[{i}] {origen}")
    print(f"     {doc.page_content[:200].strip()}")

## 7. Paso 6 — Retrieval base (Python)

Antes de llamar al LLM, Python construye el contexto base con tres fuentes:

1. **RAG search** → chunks más relevantes del vectorstore
2. **Fichero fuente** → código del objeto a testear (búsqueda por token más largo de la query)
3. **Test de referencia** → test existente del mismo proyecto como ejemplo de estilo

Este contexto base garantiza que el LLM siempre tiene información suficiente sin depender de tool calls.

In [ ]:
import os
from dataclasses import dataclass as _dataclass

# ── ProjectConfig ─────────────────────────────────────────────────────

@_dataclass
class ProjectConfig:
    """
    Describe la estructura de un proyecto AL de Business Central.
    Pensado para la integración futura con VS Code: el IDE exporta
    PROJECT_ROOT como variable de entorno para apuntar al workspace activo.
    """
    project_root: Path
    src_dirs: list
    test_dirs: list
    name: str = "BC Project"

    @classmethod
    def default(cls):
        """GarantiasAAPP hardcodeado — usado mientras no hay integración con VS Code."""
        root = Path(r"C:\Users\mscarafu\Downloads\GarantiasAAPP")
        return cls(
            project_root = root,
            src_dirs     = [root / "guarantees" / "src"],
            test_dirs    = [
                root / "guarantees-test" / "src" / "codeunit",
                root / "guarantees-test" / "test",
            ],
            name = "GarantiasAAPP",
        )

    @classmethod
    def from_path(cls, path):
        """Infiere la estructura del proyecto a partir de su raíz."""
        root = Path(path).resolve()
        src_dirs, test_dirs = [], []
        for candidate in sorted(root.rglob("*")):
            if not candidate.is_dir():
                continue
            depth = len(candidate.relative_to(root).parts)
            if depth > 3:
                continue
            name = candidate.name.lower()
            if name == "src":
                src_dirs.append(candidate)
            elif name in ("test", "tests"):
                test_dirs.append(candidate)
        return cls(project_root=root, src_dirs=src_dirs, test_dirs=test_dirs, name=root.name)

    @classmethod
    def from_env(cls):
        """Lee PROJECT_ROOT del entorno. Si no está definida, usa el default."""
        root = os.getenv("PROJECT_ROOT")
        return cls.from_path(root) if root else cls.default()

    def all_al_files(self) -> list:
        """Devuelve todos los ficheros .al del proyecto (src + tests)."""
        files = []
        for d in self.src_dirs + self.test_dirs:
            files.extend(sorted(d.rglob("*.al")))
        return files

    def find_file(self, name: str):
        """Busca un fichero .al por nombre (insensible a mayúsculas, sin extensión)."""
        name_lower = name.lower().removesuffix(".al")
        for f in self.all_al_files():
            if f.stem.lower() == name_lower:
                return f
        return None

    def is_test_file(self, path: Path) -> bool:
        """True si el fichero pertenece a alguno de los directorios de tests."""
        for test_dir in self.test_dirs:
            try:
                path.relative_to(test_dir)
                return True
            except ValueError:
                continue
        return False


# ── Constantes de truncado ────────────────────────────────────────────

RAG_CHUNK_PREVIEW  = 600
SOURCE_FILE_LIMIT  = 4_000
TEST_EXAMPLE_LIMIT = 3_000


# ── Funciones de retrieval base ───────────────────────────────────────

def _read_truncated(path: Path, limit: int) -> str:
    content   = path.read_text(encoding="utf-8", errors="replace")
    truncated = content[:limit]
    if len(content) > limit:
        truncated += f"\n// ... [truncated — {len(content)} chars total]"
    return f"// {path.name}\n\n{truncated}"


def _retrieve_rag(query: str, vectorstore, all_docs: list, k: int) -> str:
    retriever = get_retriever_for_query(vectorstore, all_docs, k)
    docs      = retriever.invoke(query)
    parts     = []
    for doc in docs:
        meta = doc.metadata
        if "chapter_num" in meta:
            header = f"[book | ch{meta['chapter_num']} \u2014 {meta['section_title']}]"
        else:
            proc   = meta.get("procedure_name") or "header"
            header = f"[{meta.get('content_type')} | {meta.get('object_type')} {meta.get('object_name')} \u2192 {proc}]"
        parts.append(f"{header}\n{doc.page_content[:RAG_CHUNK_PREVIEW]}")
    return "\n\n---\n\n".join(parts) if parts else "No results found."


def _find_source_file(query: str, config: ProjectConfig) -> str:
    tokens   = re.findall(r'[A-Za-z][A-Za-z0-9]+', query)
    al_files = [f for f in config.all_al_files() if not config.is_test_file(f)]

    # 1. Coincidencia exacta de stem (máxima prioridad)
    for token in tokens:
        for f in al_files:
            if f.stem.lower() == token.lower():
                return _read_truncated(f, SOURCE_FILE_LIMIT)

    # 2. Coincidencia parcial — token más largo primero
    for token in sorted(tokens, key=len, reverse=True):
        for f in al_files:
            if token.lower() in f.stem.lower():
                return _read_truncated(f, SOURCE_FILE_LIMIT)

    return "// Source file not found — use a tool to read the file if needed."


def _find_test_example(query: str, config: ProjectConfig) -> str:
    tokens     = re.findall(r'[A-Za-z][A-Za-z0-9]+', query)
    test_files = [f for f in config.all_al_files()
                  if config.is_test_file(f) and "library" not in f.stem.lower()]

    # Coincidencia parcial — token más largo primero
    for token in sorted(tokens, key=len, reverse=True):
        for f in test_files:
            if token.lower() in f.stem.lower():
                return _read_truncated(f, TEST_EXAMPLE_LIMIT)

    # Fallback: primer test disponible
    if test_files:
        return _read_truncated(test_files[0], TEST_EXAMPLE_LIMIT)

    return "// No test example found."


In [ ]:
# ── Uso ───────────────────────────────────────────────────────────────

config = ProjectConfig.default()
QUERY  = "Generate a test codeunit for the PostGuarantee procedure in DTNGARManagement"

rag_chunks   = _retrieve_rag(QUERY, vectorstore, all_docs, k=6)
source_code  = _find_source_file(QUERY, config)
test_example = _find_test_example(QUERY, config)

print("=== RAG (primeros 600 chars) ===")
print(rag_chunks[:600])

print("\n=== SOURCE FILE (primeros 400 chars) ===")
print(source_code[:400])

print("\n=== TEST EXAMPLE (primeros 400 chars) ===")
print(test_example[:400])

total_chars = len(rag_chunks) + len(source_code) + len(test_example)
print(f"\n=== Estimación contexto base ===")
print(f"  RAG chunks   : {len(rag_chunks):>6} chars  (~{len(rag_chunks)//4:>5} tokens)")
print(f"  Source file  : {len(source_code):>6} chars  (~{len(source_code)//4:>5} tokens)")
print(f"  Test example : {len(test_example):>6} chars  (~{len(test_example)//4:>5} tokens)")
print(f"  TOTAL        : {total_chars:>6} chars  (~{total_chars//4:>5} tokens)")

## 8. Paso 7 — Generación con el agente híbrido

`ALTestAgent` orquesta todo el proceso:
1. Construye el contexto base (Python)
2. Envía al LLM con tools disponibles — el LLM puede hacer hasta **2 tool calls extra** si necesita más información
3. Llamada final al LLM sin tools → genera el codeunit completo

**¿Por qué arquitectura híbrida?**

| Arquitectura | Ventaja | Desventaja |
|---|---|---|
| Solo RAG (Python) | Tokens predecibles, rápido | LLM no puede pedir más contexto si falta algo |
| Solo tool calling (LLM) | LLM decide qué leer | Acumula tokens en cada iteración, puede superar rate limits |
| **Híbrida (este agente)** | **Contexto base garantizado + refinamiento controlado** | **Ninguna** |

In [ ]:
import textwrap
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

# ── create_llm (llm_factory.py) ───────────────────────────────────────

def _require_key(env_var: str, provider_name: str) -> None:
    if not os.getenv(env_var):
        raise EnvironmentError(
            f"Falta la variable de entorno {env_var} para usar {provider_name}.\n"
            f"Defínela con: set {env_var}=tu_api_key  (Windows)\n"
            f"              export {env_var}=tu_api_key  (Unix)"
        )


def create_llm(provider: str, model=None, temperature: float = 0.2):
    """
    Instancia y devuelve un ChatModel de LangChain.
    Proveedores: 'openai' | 'claude' | 'ollama'
    """
    provider = provider.strip().lower()

    if provider == "openai":
        _require_key("OPENAI_API_KEY", "OpenAI")
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model or "gpt-4o", temperature=temperature)

    if provider == "claude":
        _require_key("ANTHROPIC_API_KEY", "Anthropic/Claude")
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=model or "claude-sonnet-4-6",
                             temperature=temperature, max_tokens=8192)

    if provider == "ollama":
        from langchain_ollama import ChatOllama
        return ChatOllama(model=model or "llama3", temperature=temperature)

    raise ValueError(
        f"Proveedor desconocido: {provider!r}. "
        "Opciones válidas: openai, claude, ollama."
    )


# ── AL Tools (al_tools.py) ────────────────────────────────────────────

_tool_vectorstore = None
_tool_all_docs    = []
_tool_config      = None
_tool_k           = 8


def init_tools(vs, docs: list, cfg, k: int = 8) -> None:
    """Inyecta el vectorstore, documentos y configuración en las tools."""
    global _tool_vectorstore, _tool_all_docs, _tool_config, _tool_k
    _tool_vectorstore = vs
    _tool_all_docs    = docs
    _tool_config      = cfg
    _tool_k           = k


@tool
def rag_search(query: str) -> str:
    """
    Search the knowledge base for AL testing best practices, real company
    test examples, and application source code relevant to the query.
    Use this tool first to get an overview of how similar things are tested.

    Args:
        query: Natural language description of what you are looking for.
    """
    retriever = get_retriever_for_query(
        vectorstore = _tool_vectorstore,
        all_docs    = _tool_all_docs,
        k           = _tool_k,
    )
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant results found."

    parts = []
    for doc in docs:
        meta = doc.metadata
        ct   = meta.get("content_type", "?")
        if "chapter_num" in meta:
            header = f"[book | {ct} | ch{meta['chapter_num']} \u2014 {meta['section_title']}]"
        else:
            proc   = meta.get("procedure_name") or "header"
            header = (f"[{ct} | {meta.get('object_type')} {meta.get('object_name')} "
                      f"\u2192 {proc} | {meta.get('source_file')}]")
        parts.append(f"{header}\n{doc.page_content[:800]}")

    return "\n\n---\n\n".join(parts)


@tool
def list_al_files(filter: str = "") -> str:
    """
    List all AL files in the current project (source code and tests).
    Optionally filter by a keyword that must appear in the filename.

    Args:
        filter: Optional keyword to filter filenames (case-insensitive).
    """
    if _tool_config is None:
        return "Project not configured."

    files = _tool_config.all_al_files()
    if filter:
        files = [f for f in files if filter.lower() in f.name.lower()]

    if not files:
        return f"No AL files found" + (f" matching '{filter}'." if filter else ".")

    lines = []
    for f in files:
        kind = "test" if _tool_config.is_test_file(f) else "src"
        lines.append(f"[{kind}] {f.name}  \u2192  {f}")

    return "\n".join(lines)


@tool
def read_al_file(filename: str) -> str:
    """
    Read the full content of an AL file from the current project.
    Use this tool when you need to inspect the exact code of a table,
    codeunit, page, or test file before generating a new test.

    Args:
        filename: Name of the AL file (with or without .al extension).
    """
    if _tool_config is None:
        return "Project not configured."

    path = _tool_config.find_file(filename)
    if path is None:
        all_files = _tool_config.all_al_files()
        matches   = [f for f in all_files if filename.lower() in f.name.lower()]
        if not matches:
            return (f"File '{filename}' not found. "
                    f"Use list_al_files() to see available files.")
        if len(matches) > 1:
            names = ", ".join(f.name for f in matches)
            return f"Multiple files match '{filename}': {names}. Be more specific."
        path = matches[0]

    MAX_CHARS = 6_000
    try:
        content = path.read_text(encoding="utf-8", errors="replace")
        kind    = "test" if _tool_config.is_test_file(path) else "src"
        header  = f"// [{kind}] {path.name}\n\n"
        if len(content) > MAX_CHARS:
            content = content[:MAX_CHARS] + f"\n\n// ... [truncated — {len(content)} chars total]"
        return header + content
    except OSError as e:
        return f"Error reading file: {e}"


AL_TOOLS = [rag_search, list_al_files, read_al_file]


# ── Prompts ───────────────────────────────────────────────────────────

_SYSTEM = textwrap.dedent("""\
    You are an expert in writing automated tests for Microsoft Dynamics 365 \
    Business Central using Application Language (AL).

    You will receive a base context (source code, reference test, RAG snippets).
    If you need additional information, you may call the available tools \
    (at most 2 times). Then generate the complete test codeunit.

    Rules:
    1. Produce a complete, compilable AL codeunit with Subtype = Test.
    2. Each test procedure must have the [Test] attribute.
    3. Use Assert or LibraryAssert for all assertions.
    4. Follow GIVEN / WHEN / THEN comment structure inside each test.
    5. Use helper procedures (Initialize, Create*, Verify*) to keep tests readable.
    6. Reuse the library codeunit from the reference test if it exists.
    7. Output ONLY AL code — no prose, no markdown fences.
""")

_HUMAN = textwrap.dedent("""\
    ## Source code to test
    {source_code}

    ## Reference test (same project)
    {test_example}

    ## Knowledge base snippets
    {rag_chunks}

    ## Request
    {question}
""")


# ── Loop de tool calls extra ──────────────────────────────────────────

MAX_EXTRA_CALLS = 2


def _run_extra_tool_calls(messages: list, llm_with_tools, tools_by_name: dict,
                          show_context: bool) -> list:
    """Ejecuta hasta MAX_EXTRA_CALLS rondas de tool calling."""
    for i in range(MAX_EXTRA_CALLS):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not getattr(response, "tool_calls", None):
            return messages

        for tc in response.tool_calls:
            name    = tc["name"]
            args    = tc["args"]
            tool_fn = tools_by_name.get(name)
            result  = tool_fn.invoke(args) if tool_fn else f"Unknown tool: {name}"

            if show_context:
                print(f"\n  [extra tool {i+1}] {name}({args})")
                print(f"  \u2192 {str(result)[:300]}...")

            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return messages


# ── ALTestAgent ───────────────────────────────────────────────────────

class ALTestAgent:
    def __init__(self, provider: str, model=None, persist_dir: str = DEFAULT_PERSIST,
                 device: str = "cpu", k: int = 6):
        print("[agent] Cargando modelo de embedding y vectorstore...")
        embeddings       = load_embeddings(device=device, model_name=EMBEDDING_MODEL)
        self.vectorstore = load_vectorstore(embeddings, persist_dir=persist_dir)

        raw = self.vectorstore.get(include=["documents", "metadatas"])
        self.all_docs = [
            Document(page_content=t, metadata=m)
            for t, m in zip(raw["documents"], raw["metadatas"])
        ]

        self.config = ProjectConfig.from_env()
        self.k      = k
        print(f"[agent] Proyecto  : {self.config.name}")

        llm = create_llm(provider=provider, model=model)
        init_tools(self.vectorstore, self.all_docs, self.config, k)

        self.llm_with_tools = llm.bind_tools(AL_TOOLS)
        self.llm_plain      = llm
        self.tools_by_name  = {t.name: t for t in AL_TOOLS}

    def generate(self, query: str, show_context: bool = False) -> str:
        # 1. Contexto base (Python)
        rag_chunks   = _retrieve_rag(query, self.vectorstore, self.all_docs, self.k)
        source_code  = _find_source_file(query, self.config)
        test_example = _find_test_example(query, self.config)

        if show_context:
            print(f"\n{'─'*60}\nSOURCE FILE (base)\n{'─'*60}")
            print(source_code[:400])
            print(f"\n{'─'*60}\nTEST EXAMPLE (base)\n{'─'*60}")
            print(test_example[:400])
            print(f"\n{'─'*60}\nRAG CHUNKS (base)\n{'─'*60}")
            print(rag_chunks[:600])

        human_content = _HUMAN.format(
            source_code  = source_code,
            test_example = test_example,
            rag_chunks   = rag_chunks,
            question     = query,
        )

        # 2. Tool calls extra (LLM decide si necesita más)
        messages = [
            SystemMessage(content=_SYSTEM),
            HumanMessage(content=human_content),
        ]
        messages = _run_extra_tool_calls(
            messages, self.llm_with_tools, self.tools_by_name, show_context
        )

        # 3. Generación final (sin tools)
        last = messages[-1]
        if hasattr(last, "tool_calls") and not last.tool_calls and hasattr(last, "content"):
            return last.content

        messages.append(HumanMessage(
            content="Now generate the complete AL test codeunit. Output ONLY AL code."
        ))
        response = self.llm_plain.invoke(messages)
        return response.content


# ── Inicializar el agente ─────────────────────────────────────────────

agent = ALTestAgent(
    provider    = "claude",   # "openai" | "claude" | "ollama"
    model       = None,       # usa el default del proveedor
    persist_dir = DEFAULT_PERSIST,
    device      = "cpu",
    k           = 6,
)
print("Agente listo.")

In [ ]:
QUERY = "Generate a test codeunit for the PostGuarantee procedure in DTNGARManagement"

# show_context=True muestra el contexto base y las tool calls extra en tiempo real
result = agent.generate(QUERY, show_context=True)

print("\n" + "="*70)
print("CÓDIGO AL GENERADO")
print("="*70 + "\n")
print(result)

## 9. Modo interactivo

El agente carga el modelo una sola vez y acepta queries en bucle. Útil para generar varios codeunits sin pagar el coste de recarga del modelo en cada llamada.

In [ ]:
queries = [
    "Generate a test codeunit for the PostGuarantee procedure in DTNGARManagement",
    "Generate a test codeunit for the CheckBeforePosting procedure in DTNGARGuarantee table",
]

for q in queries:
    print(f"\n{'='*70}")
    print(f"QUERY: {q}")
    print('='*70)
    code = agent.generate(q, show_context=False)
    print(code[:1000], "...")  # mostramos solo los primeros 1000 chars
    print(f"\n[{len(code.splitlines())} líneas generadas]")

## 10. Resumen del pipeline completo

| Paso | Fichero | Qué hace |
|---|---|---|
| 1. Parseo libro | `parser.py` | Markdown → secciones con `content_type` |
| 2. Chunking libro | `chunker.py` | Secciones → Documents LangChain (~2400 chars/chunk) |
| 3. Parseo AL | `al_parser.py` | Ficheros `.al` → Documents por procedimiento |
| 4. Embeddings | `embedder.py` | BGE-M3 → vectores en ChromaDB |
| 5. Retriever híbrido | `embedder.py` | MMR vectorial (60%) + BM25 (40%) |
| 6. Retrieval base | `agent.py` | RAG + fichero fuente + test referencia (Python) |
| 7. Tool calls extra | `al_tools.py` | El LLM puede pedir hasta 2 ficheros adicionales |
| 8. Generación | `agent.py` | Una llamada al LLM → codeunit AL completo |

**Configuración del proyecto** (`project_config.py`): hoy usa rutas fijas de GarantiasAAPP. En el futuro, la variable de entorno `PROJECT_ROOT` permitirá usarlo como extensión de VS Code apuntando al workspace activo.